In [1]:
# import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# Загрузка данных
train_df = pd.read_csv("/kaggle/input/vseros-otbot-5-data/train.tsv", sep='\t')
test_df = pd.read_csv('/kaggle/input/vseros-otbot-5-data/test.tsv', sep='\t')

# Проверка данных
print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")

2025-09-29 21:35:09.040246: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759181709.183859     100 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759181709.222937     100 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Размер обучающей выборки: 53494
Размер тестовой выборки: 15046


In [2]:
def split_cats(a):
    return a.split('|')
train_df['label'] = train_df['labels_str'].apply(split_cats)
train_df = train_df.explode('label')

In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63781 entries, 0 to 53493
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   app_name          63781 non-null  object
 1   full_description  63781 non-null  object
 2   shortDescription  63780 non-null  object
 3   labels_str        63781 non-null  object
 4   label             63781 non-null  object
dtypes: object(5)
memory usage: 2.9+ MB


In [4]:
train_df['text'] = (
    train_df['app_name'].fillna("") + ' ' +
    train_df['full_description'].fillna("") +
    train_df['shortDescription'].fillna("")
)

test_df['text'] = (
    test_df['app_name'].fillna("") + ' ' +
    test_df['full_description'].fillna("") +
    test_df['shortDescription'].fillna("")
)

In [5]:
# Создание маппинга для меток (если метки текстовые)
label_to_id = {label: idx for idx, label in enumerate(train_df['label'].unique())}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(label_to_id)

# Преобразование меток в числовые значения
train_df['label_id'] = train_df['label'].map(label_to_id)

# Разделение на train и validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label_id'].tolist(),
    test_size=0.05,
    random_state=42,
    stratify=train_df['label_id']
)

# Создание датасетов
def create_dataset(texts, labels=None):
    if labels is not None:
        return Dataset.from_dict({
            'text': texts,
            'labels': labels
        })
    else:
        return Dataset.from_dict({
            'text': texts
        })

# Подготовка датасетов
train_dataset = create_dataset(train_texts, train_labels)
val_dataset = create_dataset(val_texts, val_labels)
test_dataset = create_dataset(test_df['text'].tolist())

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})


In [6]:
import torch
torch.cuda.empty_cache()
# del model

In [ ]:
# Инициализация модели и токенизатора
model_name = "intfloat/multilingual-e5-large"
# model_name = "seara/rubert-tiny2-russian-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Добавление pad токена если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

# Функция токенизации
def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Применение токенизации
tokenized_datasets = dataset.map(tokenize_function, batched=True)

def hitrate_at_k(logits, y_true, k = 3):
    topk = np.argpartition(-logits, kth=k-1, axis=1)[:, :k]
    hits = 0
    for i in range(len(y_true)):
        true_idx = np.where(y_true[i] > 0)[0]
        if len(set(true_idx) & set(topk[i])) > 0:
            hits += 1
    return hits / len(y_true)

# Функция для вычисления метрик
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels_one_hot = np.eye(num_classes)[labels]
    predictions = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        "H@3": hitrate_at_k(logits, labels_one_hot, k=3)
    }

# Настройка параметров обучения
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.02,
    warmup_steps=500,
    logging_dir='./',
    logging_steps=50,
    metric_for_best_model='f1',
    greater_is_better=True,
    push_to_hub=False,
    fp16=True,
    report_to='none',  # Отключить wandb/tensorboard
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Обучение модели
print("Начинаем обучение...")
trainer.train()

# Оценка на валидационной выборке
print("\nОценка модели на валидационной выборке:")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# Сохранение модели
trainer.save_model('./final_model')
tokenizer.save_pretrained('./final_model')

# Сохранение маппинга меток
import json
with open('./final_model/label_mapping.json', 'w') as f:
    json.dump({
        'label_to_id': label_to_id,
        'id_to_label': id_to_label
    }, f)

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

Map:   0%|          | 0/15046 [00:00<?, ? examples/s]

/tmp/ipykernel_100/3319602269.py:80: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Начинаем обучение...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


In [16]:
# Генерация предсказаний для тестовой выборки
print("\nГенерация предсказаний для тестовой выборки...")
test_predictions = trainer.predict(tokenized_datasets['test'])

# Получаем вероятности для каждого класса (softmax)
probabilities = np.exp(test_predictions.predictions) / np.sum(np.exp(test_predictions.predictions), axis=1, keepdims=True)

# Для каждой строки находим индексы топ-3 классов по вероятностям
top3_indices = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]  # сортируем по убыванию

# Преобразуем индексы в текстовые метки
top3_labels = [
    [id_to_label[idx] for idx in indices]
    for indices in top3_indices
]

# Формируем строку для сабмита: "label1|label2|label3"
labels_str = ["|".join(labels) for labels in top3_labels]

# Создаём submission файл
submission = pd.DataFrame({
    'app_name': test_df['app_name'],
    'labels_str': labels_str
})


Генерация предсказаний для тестовой выборки...


In [17]:
# Сохраняем в CSV
submission.to_csv('submission_top3.tsv', index=False, sep='\t')
print(f"\nФайл submission_top3_pipe.csv создан. Количество предсказаний: {len(submission)}")


Файл submission_top3_pipe.csv создан. Количество предсказаний: 15046


In [18]:
from IPython.display import FileLink
FileLink('submission_top3.tsv')

/kaggle/working/submission_top3.tsv